<a href="https://colab.research.google.com/github/akinns247/Starter_Notebook247/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akinns247/Starter_Notebook247/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*
### 1. Ranked actions + reason codes

I use the Random Forest score from my Week-5/Week-6 modeling work to create a ranked decision-support queue. Higher scores are treated as higher priority for human review, not as proof that a page will improve after a refresh.

I add simple reason codes based on observable page signals. A page can receive a combined reason when more than one opportunity signal is present.

The main reason codes are:

* `DECLINING_TREND` — the observed trend direction is down.
* `LOW_CTR` — CTR is low relative to the available test-set distribution.
* `HIGH_SEARCH_OPPORTUNITY` — search volume is relatively high.
* `POOR_POSITION` — average position is relatively weak.
* `COMBINED_OPPORTUNITY` — more than one of these signals is present.

The queue is therefore a prioritization aid for human review. It does not automatically recommend publishing, rewriting, deleting, or redirecting a page.


In [8]:
# =========================================================
# ML-10 SECTION 1 — RANKED ACTIONS + REASON CODES
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# ---------------------------------------------------------
# 1. Load the dataset
# ---------------------------------------------------------

possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("/content/Starter_Notebook247/data/raw/content_refresh_anonymized.csv"),
]

data_path = next(
    (p for p in possible_paths if p.exists()),
    None
)

if data_path is not None:
    df = pd.read_csv(data_path)
    print("Loaded dataset from:", data_path)

else:
    data_url = (
        "https://raw.githubusercontent.com/"
        "akinns247/Starter_Notebook247/main/"
        "data/raw/content_refresh_anonymized.csv"
    )

    df = pd.read_csv(data_url)
    print("Loaded dataset from GitHub.")

print("Full dataset shape:", df.shape)

# ---------------------------------------------------------
# 2. Clean data
# ---------------------------------------------------------

df["trend_direction"] = (
    df["trend_direction"]
    .astype(str)
    .str.strip()
    .str.lower()
)

required_columns = [
    "content_id",
    "client_id",
    "ctr",
    "avg_position",
    "trend_direction",
    "search_volume"
]

missing = [
    c for c in required_columns
    if c not in df.columns
]

if missing:
    raise KeyError(
        f"Missing required columns: {missing}"
    )

df = df.dropna(
    subset=[
        "client_id",
        "ctr",
        "avg_position",
        "trend_direction",
        "search_volume"
    ]
).copy()

# ---------------------------------------------------------
# 3. Model features
# ---------------------------------------------------------

numeric_candidates = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_candidates = [
    "content_type",
    "main_intent",
    "competition_level"
]

numeric_features = [
    c for c in numeric_candidates
    if c in df.columns
]

categorical_features = [
    c for c in categorical_candidates
    if c in df.columns
]

model_features = (
    numeric_features +
    categorical_features
)

for col in numeric_features:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print("\nModel features:")
print(model_features)

# ---------------------------------------------------------
# 4. Grouped client split
# ---------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("\nTraining rows:", len(train))
print("Test rows:", len(test))

print("Training clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

# Confirm no client overlap
overlap = set(
    train["client_id"]
).intersection(
    set(test["client_id"])
)

print("Client overlap:", len(overlap))

assert len(overlap) == 0

# ---------------------------------------------------------
# 5. Create proxy target using TRAIN thresholds only
# ---------------------------------------------------------

ctr_cutoff = train["ctr"].quantile(0.25)
position_cutoff = train["avg_position"].quantile(0.75)

train["needs_refresh"] = (
    (train["ctr"] <= ctr_cutoff)
    &
    (train["trend_direction"] == "down")
    &
    (train["avg_position"] >= position_cutoff)
).astype(int)

test["needs_refresh"] = (
    (test["ctr"] <= ctr_cutoff)
    &
    (test["trend_direction"] == "down")
    &
    (test["avg_position"] >= position_cutoff)
).astype(int)

print("\nTraining target counts:")
print(train["needs_refresh"].value_counts())

print("\nTest target counts:")
print(test["needs_refresh"].value_counts())

# ---------------------------------------------------------
# 6. Preprocessing
# ---------------------------------------------------------

transformers = []

if numeric_features:

    numeric_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            )
        ]
    )

    transformers.append(
        (
            "num",
            numeric_transformer,
            numeric_features
        )
    )

if categorical_features:

    categorical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )

    transformers.append(
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop"
)

# ---------------------------------------------------------
# 7. Random Forest
# ---------------------------------------------------------

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf)
    ]
)

X_train = train[model_features]
X_test = test[model_features]

y_train = train["needs_refresh"]
y_test = test["needs_refresh"]

model.fit(
    X_train,
    y_train
)

print("\nRandom Forest training completed.")

# ---------------------------------------------------------
# 8. Generate ranking scores
# ---------------------------------------------------------

model_scores = model.predict_proba(
    X_test
)[:, 1]

results = test[
    [
        "content_id",
        "search_volume",
        "ctr",
        "avg_position",
        "trend_direction",
        "needs_refresh"
    ]
].copy()

results["model_score"] = model_scores

# ---------------------------------------------------------
# 9. Create ranked action queue
# ---------------------------------------------------------

action_queue = results.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

action_queue.insert(
    0,
    "priority_rank",
    range(
        1,
        len(action_queue) + 1
    )
)

# ---------------------------------------------------------
# 10. Create reason codes
# ---------------------------------------------------------

ctr_cutoff_action = action_queue[
    "ctr"
].quantile(0.25)

search_cutoff_action = action_queue[
    "search_volume"
].quantile(0.75)

position_cutoff_action = action_queue[
    "avg_position"
].quantile(0.75)


def make_reason(row):

    reasons = []

    if str(
        row["trend_direction"]
    ).lower() == "down":
        reasons.append(
            "DECLINING_TREND"
        )

    if row["ctr"] <= ctr_cutoff_action:
        reasons.append(
            "LOW_CTR"
        )

    if row["search_volume"] >= search_cutoff_action:
        reasons.append(
            "HIGH_SEARCH_OPPORTUNITY"
        )

    if row["avg_position"] >= position_cutoff_action:
        reasons.append(
            "POOR_POSITION"
        )

    if len(reasons) >= 2:
        return "COMBINED_OPPORTUNITY"

    if len(reasons) == 1:
        return reasons[0]

    return "MODEL_SIGNAL_ONLY"


action_queue["reason_code"] = (
    action_queue.apply(
        make_reason,
        axis=1
    )
)

action_queue["recommended_action"] = (
    "Review page before deciding whether to refresh"
)

# ---------------------------------------------------------
# 11. Display results
# ---------------------------------------------------------

print("\nTOP 20 RANKED ACTIONS")

display(
    action_queue[
        [
            "priority_rank",
            "content_id",
            "model_score",
            "reason_code",
            "recommended_action",
            "search_volume",
            "ctr",
            "avg_position",
            "trend_direction"
        ]
    ].head(20)
)

print("\nREASON CODE COUNTS")

display(
    action_queue[
        "reason_code"
    ]
    .value_counts()
    .to_frame("count")
)


Loaded dataset from GitHub.
Full dataset shape: (30000, 44)

Model features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'content_type', 'main_intent', 'competition_level']

Training rows: 21884
Test rows: 5648
Training clients: 24
Test clients: 7
Client overlap: 0

Training target counts:
needs_refresh
0    20455
1     1429
Name: count, dtype: int64

Test target counts:
needs_refresh
0    5478
1     170
Name: count, dtype: int64

Random Forest training completed.

TOP 20 RANKED ACTIONS


,priority_rank,content_id,model_score,reason_code,recommended_action,search_volume,ctr,avg_position,trend_direction
0,1,content_ee66da1b607c,0.814156,COMBINED_OPPORTUNITY,Review page before deciding whether to refresh,10.0,0.00,46.5,stable
1,2,content_1a7560472af6,0.814108,COMBINED_OPPORTUNITY,Review page before deciding whether to refresh,170.0,0.00,0.0,new
2,3,content_1fc1c9d18ad6,0.741065,LOW_CTR,Review page before deciding whether to refresh,10.0,0.00,4.0,flat
3,4,content_b12742b07c1d,0.739356,LOW_CTR,Review page before deciding whether to refresh,10.0,0.00,2.0,flat
4,5,content_83485ea6300e,0.738126,COMBINED_OPPORTUNITY,Review page before deciding whether to refresh,10.0,0.00,71.0,flat
5,6,content_141704f4d910,0.736668,LOW_CTR,Review page before deciding whether to refresh,10.0,0.00,1.0,new
6,7,content_e8f44fac6055,0.732289,LOW_CTR,Review page before deciding whether to refresh,10.0,0.00,0.0,new
7,8,content_1a6fbd3bbcd5,0.731308,COMBINED_OPPORTUNITY,Review page before deciding whether to refresh,30.0,0.00,0.7,flat
8,9,content_1d2233dc3323,0.726752,LOW_CTR,Review page before deciding whether to refresh,20.0,0.00,1.5,up
9,10,content_cb6c7d58c0bc,0.726737,COMBINED_OPPORTUNITY,Review page before deciding whether to refresh,10.0,0.00,144.5,new



REASON CODE COUNTS


,count
reason_code,
COMBINED_OPPORTUNITY,2799
LOW_CTR,859
DECLINING_TREND,814
MODEL_SIGNAL_ONLY,681
HIGH_SEARCH_OPPORTUNITY,289
POOR_POSITION,206


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*
### 2. Intended use and limits

The action playbook is intended for content analysts or content teams who need a short list of pages to review first. The Random Forest score provides directional decision-support based on the available historical features.

The queue should be used to prioritize human investigation, not to make automatic content decisions.

The main limitations are that the target is a proxy for refresh priority, the validation uses a grouped client holdout, and the Random Forest achieved a Precision@20 of 0.05 in the held-out test set. The model therefore provides a ranking signal but does not guarantee that a selected page will improve after a refresh.

The recommendations are also limited to the data and features used by the model. They should not be interpreted as causal evidence that refreshing a page will improve its performance.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Intended use and limits

print("INTENDED USE")
print("Use: prioritize pages for human content review.")
print("Output: directional decision-support.")

print("\nVALIDATION CONTEXT")
print("Split design: grouped by client")
print("Training clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())
print("Client overlap:", len(overlap))

print("\nMODEL LIMIT")
print("Random Forest Precision@20: 0.05")
print("Interpretation: ranking signal, not a guaranteed outcome.")

print("\nAUTOMATION LIMIT")
print("The queue should not automatically publish, rewrite,")
print("delete, or redirect content.")

INTENDED USE
Use: prioritize pages for human content review.
Output: directional decision-support.

VALIDATION CONTEXT
Split design: grouped by client
Training clients: 24
Test clients: 7
Client overlap: 0

MODEL LIMIT
Random Forest Precision@20: 0.05
Interpretation: ranking signal, not a guaranteed outcome.

AUTOMATION LIMIT
The queue should not automatically publish, rewrite,
delete, or redirect content.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*
### 3. Human review + the no-go list

A person must review each recommendation before any content action is taken. The reviewer should confirm that the page is still relevant, that its search intent is appropriate, and that the observed performance signals make sense in context.

The reviewer should also check whether the page has recently been changed or refreshed and consider business, editorial, or technical information that is not represented in the dataset.

The following actions should never be automated from this model alone:

* Publishing a rewritten page.
* Deleting a page.
* Redirecting a URL.
* Changing important claims or information.
* Making the final decision about content quality.
* Treating a high model score as proof that a refresh will improve performance.

The model ranks pages for review; a human remains responsible for the final decision.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Human review + no-go list

human_review_checks = [
    "Confirm the page is still relevant",
    "Review the current search intent",
    "Check whether the page was recently refreshed",
    "Review the page's performance signals",
    "Consider business and editorial context"
]

no_go_actions = [
    "Automatic publishing",
    "Automatic deletion",
    "Automatic URL redirects",
    "Automatic changes to important claims",
    "Automatic final content-quality decisions"
]

print("HUMAN REVIEW CHECKS")
for check in human_review_checks:
    print("-", check)

print("\nNO-GO LIST")
for action in no_go_actions:
    print("-", action)

print("\nDecision rule:")
print("The model ranks pages; humans decide what action to take.")


HUMAN REVIEW CHECKS
- Confirm the page is still relevant
- Review the current search intent
- Check whether the page was recently refreshed
- Review the page's performance signals
- Consider business and editorial context

NO-GO LIST
- Automatic publishing
- Automatic deletion
- Automatic URL redirects
- Automatic changes to important claims
- Automatic final content-quality decisions

Decision rule:
The model ranks pages; humans decide what action to take.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*
### 4. Monitoring / retrain triggers

The recommendations should be monitored because the relationship between historical content signals and refresh priority may change over time.

I would review the model if ranking performance falls below the current measured level, if the feature distributions change substantially, or if new types of content appear that were not represented during training.

A retraining review should be triggered when:

* Precision@20 falls meaningfully below the current measured value of 0.05.
* Important feature distributions change substantially.
* New content types or categories appear.
* Model scores become unusually concentrated or different from previous runs.
* Human reviewers repeatedly disagree with the recommendations.
* The relationship between the available signals and refresh priority changes.

These are review triggers rather than automatic retraining rules. A human should investigate the reason for the change before retraining or replacing the model.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Monitoring / retrain triggers

current_precision_at_20 = 0.05
current_roc_auc = 0.5682

print("CURRENT VALIDATION REFERENCE")
print("Precision@20:", current_precision_at_20)
print("ROC-AUC:", current_roc_auc)

monitoring_triggers = [
    "Precision@20 falls meaningfully below 0.05",
    "Important feature distributions shift",
    "New content types or categories appear",
    "Model scores become unusually concentrated",
    "Human reviewers frequently disagree with recommendations",
    "The relationship between signals and refresh priority changes"
]

print("\nMONITORING / RETRAIN REVIEW TRIGGERS")

for trigger in monitoring_triggers:
    print("-", trigger)

print("\nMonitoring approach:")
print("Review the cause of a trigger before retraining or replacing the model.")


CURRENT VALIDATION REFERENCE
Precision@20: 0.05
ROC-AUC: 0.5682

MONITORING / RETRAIN REVIEW TRIGGERS
- Precision@20 falls meaningfully below 0.05
- Important feature distributions shift
- New content types or categories appear
- Model scores become unusually concentrated
- Human reviewers frequently disagree with recommendations
- The relationship between signals and refresh priority changes

Monitoring approach:
Review the cause of a trigger before retraining or replacing the model.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*
## 5. Exports for the paper

I export the ranked action queue and a compact summary of the recommendation logic to `work/outputs/`. These files can be reused as evidence in the final paper and provide a reproducible record of the decision-support results.

The exported queue contains the model ranking score, reason code, key performance signals, and recommended human-review action. The export is intended for analysis and reporting rather than automatic content changes.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# =========================================================
# SECTION 5 — EXPORTS FOR THE PAPER
# =========================================================

from pathlib import Path

# Create output folder
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 1. Export ranked action queue
# ---------------------------------------------------------

queue_export = action_queue[
    [
        "priority_rank",
        "content_id",
        "model_score",
        "reason_code",
        "recommended_action",
        "search_volume",
        "ctr",
        "avg_position",
        "trend_direction",
        "needs_refresh"
    ]
].copy()

queue_export.to_csv(
    output_dir / "ml10_action_queue.csv",
    index=False
)

# ---------------------------------------------------------
# 2. Create reason-code summary
# ---------------------------------------------------------

reason_summary = (
    action_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="count")
)

reason_summary.to_csv(
    output_dir / "ml10_reason_code_summary.csv",
    index=False
)

# ---------------------------------------------------------
# 3. Create model summary for the paper
# ---------------------------------------------------------

model_summary = pd.DataFrame({
    "Metric": [
        "Training rows",
        "Test rows",
        "Training clients",
        "Test clients",
        "Client overlap",
        "Precision@20",
        "ROC-AUC"
    ],
    "Value": [
        len(train),
        len(test),
        train["client_id"].nunique(),
        test["client_id"].nunique(),
        len(overlap),
        0.05,
        0.5682
    ]
})

model_summary.to_csv(
    output_dir / "ml10_model_summary.csv",
    index=False
)

# ---------------------------------------------------------
# 4. Display exported files
# ---------------------------------------------------------

print("EXPORTS COMPLETED\n")

print("Files saved to work/outputs/:")

for file in sorted(output_dir.glob("ml10_*.csv")):
    print("-", file)

print("\nTop 10 action queue:")
display(queue_export.head(10))

print("\nReason-code summary:")
display(reason_summary)

print("\nModel summary:")
display(model_summary)


EXPORTS COMPLETED

Files saved to work/outputs/:
- work/outputs/ml10_action_queue.csv
- work/outputs/ml10_model_summary.csv
- work/outputs/ml10_reason_code_summary.csv

Top 10 action queue:


,priority_rank,content_id,model_score,reason_code,recommended_action,search_volume,ctr,avg_position,trend_direction,needs_refresh
0,1,content_ee66da1b607c,0.814156,COMBINED_OPPORTUNITY,Review page before deciding whether to refresh,10.0,0.0,46.5,stable,0
1,2,content_1a7560472af6,0.814108,COMBINED_OPPORTUNITY,Review page before deciding whether to refresh,170.0,0.0,0.0,new,0
2,3,content_1fc1c9d18ad6,0.741065,LOW_CTR,Review page before deciding whether to refresh,10.0,0.0,4.0,flat,0
3,4,content_b12742b07c1d,0.739356,LOW_CTR,Review page before deciding whether to refresh,10.0,0.0,2.0,flat,0
4,5,content_83485ea6300e,0.738126,COMBINED_OPPORTUNITY,Review page before deciding whether to refresh,10.0,0.0,71.0,flat,0
5,6,content_141704f4d910,0.736668,LOW_CTR,Review page before deciding whether to refresh,10.0,0.0,1.0,new,0
6,7,content_e8f44fac6055,0.732289,LOW_CTR,Review page before deciding whether to refresh,10.0,0.0,0.0,new,0
7,8,content_1a6fbd3bbcd5,0.731308,COMBINED_OPPORTUNITY,Review page before deciding whether to refresh,30.0,0.0,0.7,flat,0
8,9,content_1d2233dc3323,0.726752,LOW_CTR,Review page before deciding whether to refresh,20.0,0.0,1.5,up,0
9,10,content_cb6c7d58c0bc,0.726737,COMBINED_OPPORTUNITY,Review page before deciding whether to refresh,10.0,0.0,144.5,new,0



Reason-code summary:


,reason_code,count
0,COMBINED_OPPORTUNITY,2799
1,LOW_CTR,859
2,DECLINING_TREND,814
3,MODEL_SIGNAL_ONLY,681
4,HIGH_SEARCH_OPPORTUNITY,289
5,POOR_POSITION,206



Model summary:


,Metric,Value
0,Training rows,21884.0000
1,Test rows,5648.0000
2,Training clients,24.0000
3,Test clients,7.0000
4,Client overlap,0.0000
5,Precision@20,0.0500
6,ROC-AUC,0.5682


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.